In [1]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
import sys
import os

# Add the directory containing the file to the system path
sys.path.append(os.path.abspath("../module1"))

from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

Running the agent

Reuse the ToyAIKit agent from module 01. It handles the agent loop and stores the full message history.

First, set up the model clients:

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from toyaikit.llm import OpenAIClient

load_dotenv()
openai_client = OpenAI()

Define the search tool:

In [6]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

Create the runner:

In [7]:
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner

agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

In [8]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

In [10]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='I just found this course, is it too late for me to join, or can I still sign up?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"too late to join sign up enrollment deadline late registration course FAQ"}', call_id='call_hFiHnZLZe4uMgZ6zcLkUTFzL', name='search', type='function_call', id='fc_03c177195bb73e66006a517be0fc3c81a199570027989a999d', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_hFiHnZLZe4uMgZ6zcLkUTFzL',
  'output': '[\n  {\n    "id": "cdc3b285e5",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Can I submit homework after the deadline, or get a deadline extension?",\n    "answer": "No. We don\'t give individual dead

For this lesson, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

Extract the function name and arguments:



In [11]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls

In [12]:
tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"too late to join sign up enrollment deadline late registration course FAQ"}'}]

In [13]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [18]:
original_doc, answer_orig

({'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.')

In [19]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'I just found this course, is it too late for me to join, or can I still sign up?',
 'answer_agent': 'Yes — you can still join.\n\nAccording to the course FAQ, if you just discovered the course, you’re still welcome to start. If you want a certificate, make sure you submit your project while submissions are still being accepted. Also, you can begin learning and submitting homework while the form is open, even if the deadline has already passed.\n\nIf you want, I can also help you figure out whether the homework/project submission window is still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"too late to join sign up enrollment deadline late registration course FAQ"}'}],
 'cost': Decimal('0.00117975'),
 'document': '74eb249bbf'}

Processing multiple questions

In [20]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

Run it for a small sample in parallel:

In [21]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

Turn it into a dataframe:

In [22]:
df_agent = pd.DataFrame(agent_answers)

In [23]:
df_agent["cost"].sum()

Decimal('0.06530625')

In [24]:
df_agent.to_csv("data/agent-answers.csv", index=False)

In [25]:
df_agent = pd.read_csv("data/agent-answers.csv")
agent_answers = df_agent.to_dict(orient="records")

Judging answers and trajectories

A good trajectory is not just "many tool calls". A good trajectory uses the available tools in a way that helps answer the question.


For our search agent, a good trajectory has these properties:

*The search query is relevant to the user question

*The query includes the important keywords from the question

*The agent avoids duplicate searches with the same arguments

*If it searches more than once, the next query is a useful refinement

*It usually uses 1 search call

*2-3 calls can be okay for harder questions

*More than 3 search calls needs a clear reason

*The tool calls support the final answer

*The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:



In [26]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

The judge instructions:

In [27]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

Define the judge function:

In [36]:
import json
import ast
from evaluation_utils import calc_total_price, llm_structured_retry

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        try:
            # First, try standard JSON parsing
            tool_calls = json.loads(tool_calls)
        except json.JSONDecodeError:
            # If that fails, assume it's a Python-formatted string (single quotes)
            tool_calls = ast.literal_eval(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [37]:
agent_eval, usage = evaluate_agent_answer(agent_answers[0])

agent_eval

AgentEvaluation(answer_reasoning="The agent's answer matches the ground truth. It clearly says the user can still join, and it includes the important condition that to receive a certificate, the project must be submitted while submissions are still being accepted.", answer_score='good', trajectory_reasoning="The search query was relevant to the user's question about whether it is too late to join or sign up. It used appropriate keywords about late enrollment and signing up, and only one search was needed. The tool use was reasonable and supported the final answer.", trajectory_score='good')

When the answer is bad, the trajectory score tells us whether the problem started with tool use. If the answer is bad but the trajectory is good, the model may have used the retrieved context poorly. If both are bad, the agent likely searched for the wrong thing. It may also have stopped too early.

Running the agent judge


Run the judge for all agent answers:


In [38]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

Use the same parallel helper:

In [39]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

Split the results:

In [40]:
agent_evaluations = []
usages = []

for evaluation, usage in results:
    agent_evaluations.append(evaluation)
    usages.append(usage)

Create a dataframe:

In [41]:
df_agent_eval = pd.DataFrame(agent_evaluations)

In [42]:
calc_total_price(usages)

0.05466825000000001

In [43]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    45
bad      5
Name: count, dtype: int64

In [44]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [45]:
df_agent_eval.to_csv("data/agent-evaluations.csv", index=False)